# PID tuning classifier — interactive walkthrough

This notebook mirrors **Phase C** (`ml/train_tuning_classifier.py`): load `ml/data/tuning_runs.csv`, train baselines + Random Forest, plot class balance, confusion matrix, and feature importances.

**Working directory:** start Jupyter from the **repository root** (`ecu-simulation-dashboard/`), or adjust `ROOT` in the next cell so it points at that folder.

### Optional: notebook dependencies

Uncomment if `pandas` / `matplotlib` are missing in your environment.

In [ ]:
# %pip install pandas matplotlib -q

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path(os.getcwd()).resolve()
if not (ROOT / "backend").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "backend").is_dir():
    raise RuntimeError(
        "Could not find project root (folder with backend/). "
        "Start Jupyter from the ecu-simulation-dashboard directory, or set ROOT manually."
    )
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

## Load data and class balance

In [ ]:
import pandas as pd

DATA = ROOT / "ml/data/tuning_runs.csv"
df = pd.read_csv(DATA)
df.head()

In [ ]:
df["label"].value_counts().sort_index()

In [ ]:
import matplotlib.pyplot as plt

df["label"].value_counts().sort_index().plot(kind="bar", title="Class counts (training labels)", figsize=(9, 3.5))
plt.ylabel("count")
plt.tight_layout()
plt.show()

## Train & evaluate (same logic as `python ml/train_tuning_classifier.py`)

Sets `save_artifacts=True` to refresh `ml/artifacts/` (joblib + metrics.json). Use `save_artifacts=False` for quick plots only.

In [ ]:
from ml.train_tuning_classifier import train_and_save

result = train_and_save(
    DATA,
    ROOT / "ml/artifacts",
    test_size=0.2,
    seed=42,
    save_artifacts=True,
)
m = result["metrics"]
print("Rule replay (full CSV):     ", m["rule_replay_accuracy_full"])
print("Val acc — majority baseline:", m["baseline_majority_val_accuracy"])
print("Val acc — logistic:        ", m["logistic_val_accuracy"])
print("Val acc — random forest:   ", m["random_forest_val_accuracy"])

## Confusion matrix (validation, Random Forest)

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ConfusionMatrixDisplay(
    confusion_matrix=np.asarray(result["confusion_matrix"]),
    display_labels=result["labels_sorted"],
).plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title("Random Forest — validation set")
plt.tight_layout()
plt.show()

## Random Forest feature importances

In [ ]:
from ml.train_tuning_classifier import FEATURE_COLS

imp = result["feature_importances"]
order = np.argsort(-imp)
plt.figure(figsize=(8, 5))
plt.barh([FEATURE_COLS[i] for i in order[::-1]], imp[order[::-1]], color="steelblue")
plt.xlabel("importance")
plt.title("Random Forest feature importances")
plt.tight_layout()
plt.show()

Per-class precision/recall (validation) are in `m["classification_report_val_rf"]` and in `ml/artifacts/metrics.json` after training.